# EasyOCR experiment

Standalone notebook to try EasyOCR against the real tag photos in `tests/fixtures/`, compare it to the current Tesseract pipeline, and experiment freely. Nothing here touches the app -- it's a sandbox.

**Kernel:** select "Python (igi-ocr)" (the project's conda env -- `easyocr`, `jupyter`, and everything else are already installed there).

**Findings so far** (see below for the code that produced these):
- EasyOCR's raw text recognition is noticeably better than Tesseract's on these photos (REPORT/CVD correct 6/6 vs 3/6, carat content correct 6/6, shape correct 6/6).
- It's slow on CPU: roughly 30-40 seconds per image on this machine. Tesseract takes about a second.
- It pulls in PyTorch + torchvision + scipy + scikit-image -- hundreds of MB, versus Tesseract's tiny footprint. That's a real risk for Streamlit Community Cloud's free tier (build size / memory limits).
- The existing `parsing.py` regexes are tuned for Tesseract's output shape and don't perfectly fit EasyOCR's per-text-box output (color/clarity in particular needs the two boxes merged onto one line, and a couple of small format mismatches like `"1"` read as `"T"`).

## Setup

In [ ]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"  # avoids a Windows console crash in EasyOCR's progress bar

import time
from pathlib import Path

import cv2
import easyocr
import matplotlib.pyplot as plt

import ocr as tesseract_ocr  # the project's existing Tesseract wrapper (ocr.py)
import parsing

FIXTURES_DIR = Path("tests/fixtures")

# Ground truth read directly off each tag photo, for scoring.
CASES = [
    ("sample_tag.jpeg", "809614206", "CVD", "EMERALD", "3.01", "E", "VS1"),
    ("tag_e79422_marquise.jpeg", "791655400", "CVD", "MARQUISE", "2.04", "D", "VVS2"),
    ("tag_c141640_heart.jpeg", "817630270", "CVD", "HEART", "1.00", "F", "VS1"),
    ("tag_e84454_emerald.jpeg", "804633671", "CVD", "EMERALD", "2.97", "E", "VVS1"),
    ("tag_c141641_heart.jpeg", "817634109", "CVD", "HEART", "1.00", "F", "VS1"),
    ("tag_e86943_oval.jpeg", "809609517", "CVD", "OVAL", "1.00", "D", "VVS1"),
]
FIELDS = ["report_type", "shape", "carat", "color", "clarity"]

## Load the EasyOCR reader

First run downloads the detection + recognition model weights (a few minutes). After that it's cached locally and loads fast.

In [ ]:
reader = easyocr.Reader(["en"], gpu=False, verbose=False)
print("EasyOCR reader ready")

## Look at one photo

Change `filename` to try a different fixture, or point it at any other image path (e.g. one of your own cropped photos).

In [ ]:
filename = "sample_tag.jpeg"
image = cv2.imread(str(FIXTURES_DIR / filename), cv2.IMREAD_COLOR)

plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(filename)
plt.show()

## Run EasyOCR on it (with timing)

`detail=1` returns `(bbox, text, confidence)` per detected text box -- this is what lets us see roughly where on the tag each piece of text was found, which matters for reconstructing lines (see next cell).

In [ ]:
t0 = time.time()
detections = reader.readtext(image, detail=1)
print(f"{time.time() - t0:.1f}s")

for bbox, text, conf in detections:
    print(f"{conf:.2f}  {text!r}")

## Reconstruct text lines from the detected boxes

EasyOCR returns one box per detected text fragment, not one continuous text block like Tesseract. To feed the existing line-based `parsing.parse_fields`, boxes that sit on roughly the same row get grouped into one line (left-to-right). `y_tolerance` controls how close two boxes' vertical centers need to be to count as "the same row" -- tune it if lines are merging wrong or splitting wrong for a given photo.

In [ ]:
def group_into_lines(detections, y_tolerance=15):
    items = []
    for bbox, text, conf in detections:
        ys = [p[1] for p in bbox]
        xs = [p[0] for p in bbox]
        items.append((sum(ys) / len(ys), min(xs), text))
    items.sort(key=lambda t: t[0])

    lines, current_line, current_y = [], [], None
    for y, x, text in items:
        if current_y is None or abs(y - current_y) <= y_tolerance:
            current_line.append((x, text))
            current_y = y if current_y is None else (current_y + y) / 2
        else:
            lines.append(" ".join(t for _, t in sorted(current_line)))
            current_line, current_y = [(x, text)], y
    if current_line:
        lines.append(" ".join(t for _, t in sorted(current_line)))
    return "\n".join(lines)


raw_text = group_into_lines(detections)
print(raw_text)

## Parse it with the existing field parser

In [ ]:
fields = parsing.parse_fields(raw_text)
fields

## Compare EasyOCR vs Tesseract across all 6 real photos

This is the same scoring approach used earlier in the conversation: for each field, count correct / wrong / missing against the known ground truth. Re-run after changing `group_into_lines` (e.g. its `y_tolerance`) or after editing `parsing.py` to see the effect immediately.

In [ ]:
import imaging  # the project's current OpenCV preprocessing, for the Tesseract side


def score(label, get_raw_text_fn):
    correct = wrong = missing = 0
    total_time = 0.0
    rows = []
    for filename, igi, *expected in CASES:
        img = cv2.imread(str(FIXTURES_DIR / filename), cv2.IMREAD_COLOR)
        t0 = time.time()
        text = get_raw_text_fn(img)
        elapsed = time.time() - t0
        total_time += elapsed
        parsed = parsing.parse_fields(text)
        for field, exp in zip(FIELDS, expected):
            actual = parsed.get(field)
            if actual is None:
                missing += 1
            elif actual == exp:
                correct += 1
            else:
                wrong += 1
        rows.append((filename, elapsed, parsed))
    total = len(CASES) * len(FIELDS)
    print(f"{label}: {correct}/{total} correct, {wrong} wrong, {missing} missing | avg {total_time/len(CASES):.1f}s/image")
    return rows


def tesseract_raw_text(img):
    return tesseract_ocr.run_ocr(imaging.preprocess(img))


def easyocr_raw_text(img):
    return group_into_lines(reader.readtext(img, detail=1))


tesseract_rows = score("Tesseract (current pipeline)", tesseract_raw_text)
easyocr_rows = score("EasyOCR", easyocr_raw_text)

## Per-photo detail

Handy for seeing exactly what each engine parsed out of each photo, side by side.

In [ ]:
for (filename, t_time, t_fields), (_, e_time, e_fields) in zip(tesseract_rows, easyocr_rows):
    print(f"=== {filename} ===")
    print(f"  Tesseract ({t_time:.1f}s): {t_fields}")
    print(f"  EasyOCR   ({e_time:.1f}s): {e_fields}")
    print()

## Try your own photo

Drop an image anywhere under the project and point this at it -- e.g. a fresh crop you want to test.

In [ ]:
my_photo_path = "tests/fixtures/sample_tag.jpeg"  # <-- change this

img = cv2.imread(my_photo_path, cv2.IMREAD_COLOR)
plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

t0 = time.time()
text = easyocr_raw_text(img)
print(f"EasyOCR: {time.time() - t0:.1f}s")
print(text)
print()
print("Parsed:", parsing.parse_fields(text))